# Import Packages

In [146]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [147]:
import sys
import requests
import pandas as pd
# import geopandas as gpd
from shapely.geometry import Point, Polygon
import folium
import json
import time
import numpy as np
import h3
from folium.plugins import HeatMap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
import time
import json

In [148]:
sys.path.append('src')

In [149]:
from fetch_data import *
from data_io import *
from data_prep import *
from scoring import *
from threshold_clustering import *
from dbscan_clustering import *
from visualization import *

# Inputs

In [150]:
#### FRONTEND INPUTS ####

#the UI will instead give ranks, we need to convert to weights
user_weights = {
    'police_station': 1,
    'grocery_store': 1,
    'hospital': 1,
    'marta_stop': 2,
    'school': 1,
    'restaurant': 1,
    'park': 1
}

# radius in miles
user_radius_miles = 12/1.60934
user_has_vehicle = True
center = (33.749, -84.388)


# convert to km
user_radius_km = user_radius_miles * 1.60934 

# Query Data

In [151]:
# all_pois = []
# all_pois = query_restaurant_data(ATLANTA_BBOX, all_pois)
# all_pois = query_park_data(ATLANTA_BBOX, all_pois)
# all_pois = query_hospital_and_clinic_data(ATLANTA_BBOX, all_pois)
# print(f"Total POIs fetched: {len(all_pois)}")

# Query Save & Load

In [152]:
name_of_the_file = "combine_datasets_v2"

In [153]:
# save_pois(all_pois, name_of_the_file)

In [154]:
df_pois = load_pois(name_of_the_file)

Loaded 170681
Summary by type:
type
school            102130
crime_incident     50202
marta_stop          9266
hospital            8013
park                 503
grocery_store        300
police_station       247
restaurant            20
Name: count, dtype: int64


# Basic EDA

In [155]:
df_pois.isnull().sum(axis=0)

type    0
name    0
lat     0
lon     0
dtype: int64

In [156]:
df_pois.groupby('type').agg({'lat':['min','max'] , 'lon':['min','max'], 'type':['count']})
#looks like the data has higher coverage than just Atlanta and Metro Atlanta

lat                    lon                type
                      min        max         min         max   count
type                                                                
crime_incident  32.151695  34.777687  -85.711189  -82.680045   50202
grocery_store   33.293267  34.261048  -84.888640  -83.886021     300
hospital       -14.290242  71.297725 -176.640263  145.724472    8013
marta_stop      33.432372  34.105822  -84.669803  -84.083455    9266
park            33.453098  33.976909  -84.552848  -84.153698     503
police_station  32.845390  34.557793  -85.287257  -83.596301     247
restaurant      33.761162  33.859863  -84.455464  -84.329030      20
school         -14.348924  71.300337 -176.640331  145.784430  102130

In [157]:
df_pois = keep_pois_within_bbox(df_pois, user_radius_km)
df_pois.head()

Filtered POIs from 170681 to 57626 within bbox


,type,name,lat,lon
0,police_station,DEKALB TECHNICAL COLLEGE POLICE,33.789938,-84.234406
1,police_station,DEKALB COUNTY SHERIFFS OFFICE / DEKALB COUNTY ...,33.775744,-84.244527
2,police_station,DEKALB COUNTY MARSHALS OFFICE,33.774070,-84.297214
3,police_station,GEORGIA BUREAU OF INVESTIGATION,33.692882,-84.272514
4,police_station,FULTON COUNTY MARSHALS OFFICE,33.750652,-84.391145


In [158]:
df_pois.groupby('type').agg({'lat':['min','max'] , 'lon':['min','max'], 'type':['count']})

lat                   lon              type
                      min        max        min        max  count
type                                                             
crime_incident  33.624855  33.878633 -84.543773 -84.252012  50129
grocery_store   33.620239  33.876639 -84.531024 -84.239643     62
hospital        33.680272  33.857813 -84.513144 -84.248184     25
marta_stop      33.619354  33.878604 -84.543062 -84.232170   6642
park            33.654428  33.876902 -84.538924 -84.284425    497
police_station  33.620332  33.849226 -84.540764 -84.234406     55
restaurant      33.761162  33.859863 -84.455464 -84.329030     20
school          33.621900  33.875800 -84.538900 -84.232900    196

In [159]:
df_pois['type'].unique()

array(['police_station', 'grocery_store', 'hospital', 'marta_stop',
       'school', 'restaurant', 'crime_incident', 'park'], dtype=object)

# Data Prep - Data Points

In [160]:
# either use boundaries or radius function to create hex grids
# hexagons = create_hex_grids_with_boundaries(df_pois)
hexagons = create_hex_grids_with_radius(df_pois, radius_km=user_radius_km, center = center, size_of_grid = 8)



Using circular boundary: center (33.7490, -84.3880), radius 12.0 km
Generated 890 hexagons (before filtering)
Filtered to 536 hexagons within 12.0 km of center


# Data Prep - Features

In [161]:
## define config for each POI type
# sameple config, the below dictionary represents the default settings for each POI type. For custom seetings, modify and pass it to calculate_accessibility_scores function

# poi_types_config = {
#     'restaurant': {'types': ['restaurant'], 'decay_rate': 1.5, 'max_distance_km': 10, 'invert': False},
#     'grocery_store': {'types': ['grocery_store'], 'decay_rate': 2, 'max_distance_km': 8, 'invert': False},
#     'school': {'types': ['school'], 'decay_rate': 1, 'max_distance_km': 15, 'invert': False},
#     'hospital': {'types': ['hospital'], 'decay_rate': 0.8, 'max_distance_km': 20, 'invert': False},
#     'marta_stop': {'types': ['marta_stop'], 'decay_rate': 0.5, 'max_distance_km': 5, 'invert': False},
#     'police_station': {'types': ['police_station'], 'decay_rate': 0.5, 'max_distance_km': 10, 'invert': False},
#     'park': {'types': ['park'], 'decay_rate': 1.0, 'max_distance_km': 5, 'invert': False},
#     'crime_incident': {'types': ['crime_incident'], 'decay_rate': 2.0, 'max_distance_km': 3, 'invert': True},
# }

In [162]:
# will take some time to run
# pass user_has_vehicle to the function later
df_hexagons = calculate_accessibility_scores(hexagons, df_pois, user_has_vehicle)

>> Using cached accessibility scores


In [163]:
#lot's of MARTA stops - sanity check
df_hexagons = calculate_accessibility_scores(hexagons, df_pois, user_has_vehicle)

>> Using cached accessibility scores


In [164]:
# df_hexagons = apply_user_weights(df_hexagons, user_weights)


df_hexagons = apply_user_weights(df_hexagons, user_weights, smooth_before_weighting=True, neighbor_weight=0.3)



print(df_hexagons.head())

Applying user preferences: {'police_station': 1, 'grocery_store': 1, 'hospital': 1, 'marta_stop': 2, 'school': 1, 'restaurant': 1, 'park': 1}
Normalized weights (exponential): {'police_station': 0.15384615384615385, 'grocery_store': 0.15384615384615385, 'hospital': 0.15384615384615385, 'marta_stop': 0.07692307692307693, 'school': 0.15384615384615385, 'restaurant': 0.15384615384615385, 'park': 0.15384615384615385}

Applying spatial smoothing to 7 score columns...
  Smoothing hexagon 0/536...
  Smoothing hexagon 50/536...
  Smoothing hexagon 100/536...
  Smoothing hexagon 150/536...
  Smoothing hexagon 200/536...
  Smoothing hexagon 250/536...
  Smoothing hexagon 300/536...
  Smoothing hexagon 350/536...
  Smoothing hexagon 400/536...
  Smoothing hexagon 450/536...
  Smoothing hexagon 500/536...
Spatial smoothing complete

User Match Score Statistics:
count    536.000000
mean       0.377197
std        0.171016
min        0.069277
25%        0.246296
50%        0.363578
75%        0.49628

In [165]:
print(df_hexagons['user_match_score'].describe())

count    536.000000
mean       0.377197
std        0.171016
min        0.069277
25%        0.246296
50%        0.363578
75%        0.496284
max        0.850106
Name: user_match_score, dtype: float64


In [166]:
# Top 5 hexagons
top_5 = df_hexagons.nlargest(5, 'user_match_score')
print(top_5[['hex_id', 'user_match_score', 'restaurant_accessibility', 'grocery_store_accessibility']])

# Bottom 5 hexagons
bottom_5 = df_hexagons.nsmallest(5, 'user_match_score')
print(bottom_5[['hex_id', 'user_match_score', 'restaurant_accessibility', 'grocery_store_accessibility']])

              hex_id  user_match_score  restaurant_accessibility  \
244  8844c1a8e1fffff          0.850106                  3.471225   
183  8844c1a8e7fffff          0.821473                  3.278217   
174  8844c1a8e3fffff          0.817678                  3.528258   
280  8844c1a9dbfffff          0.811812                  2.631071   
353  8844c1a8e5fffff          0.806777                  2.814433   

     grocery_store_accessibility  
244                     1.836989  
183                     1.883489  
174                     1.816818  
280                     2.327211  
353                     1.705596  
              hex_id  user_match_score  restaurant_accessibility  \
495  8844c1ab61fffff          0.069277                  0.000000   
207  8844c1ab45fffff          0.075073                  0.000000   
112  8844c1ab67fffff          0.075828                  0.002145   
147  8844c1ab6bfffff          0.083911                  0.001647   
361  8844c1ab49fffff          0.084596   

# Experiment 1: Threshold Clustering

In [167]:
# df_threshold = cluster_based_on_score(df_hexagons)
# df_threshold.head()


df_classified = cluster_based_on_score(
    df_hexagons,
    n_tiers=10
)

Classifying into 10 tiers:
  Tier 0 threshold (top 90.0%): 0.612
  Tier 1 threshold (top 80.0%): 0.525
  Tier 2 threshold (top 70.0%): 0.463
  Tier 3 threshold (top 60.0%): 0.406
  Tier 4 threshold (top 50.0%): 0.364
  Tier 5 threshold (top 40.0%): 0.321
  Tier 6 threshold (top 30.0%): 0.271
  Tier 7 threshold (top 20.0%): 0.217
  Tier 8 threshold (top 10.0%): 0.160

Suitability Distribution:
suitability_label
Less Suitable    53
Most Suitable    54
Okay             54
Name: count, dtype: int64

SUITABILITY TIER CHARACTERISTICS

Most Suitable (54 hexagons):
  Match Score Range: 0.613 - 0.850
  Avg restaurant: 2.205
  Avg grocery_store: 1.381
  Avg school: 17.413

Okay (54 hexagons):
  Match Score Range: 0.525 - 0.611
  Avg restaurant: 1.378
  Avg grocery_store: 0.893
  Avg school: 16.913

Less Suitable (53 hexagons):
  Match Score Range: 0.463 - 0.525
  Avg restaurant: 1.018
  Avg grocery_store: 0.808
  Avg school: 16.417


In [168]:
threshold_map_name = "data/output_data/atlanta_threshold_map.html"
map_threshold = create_suitability_map(df_classified, user_weights)
map_threshold.save(threshold_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...


In [169]:
save_csv(df_classified, "combined_data_outputs")

Saved CSV: data/output_data/combined_data_outputs.csv


In [170]:
df_classified_json = convert_json(df_classified)

In [171]:
save_json(df_classified_json, "combined_data_outputs_json")

# Experiment 2: DBSCAN Clustering

In [172]:
df_dbscan = dbscan_score_clustering(df_hexagons, eps=0.1, min_samples=3)


DBSCAN CLUSTERING (Score-Based)
Parameters: eps=0.1, min_samples=3
Score range after scaling: [0.000, 1.000]

Results:
  Clusters found: 1
  Noise points: 0
  Cluster 0: 536 hexagons, avg score = 0.377


In [173]:
cluster_colors = get_cluster_colors(df_dbscan)

In [174]:
dbscan_map_name = "data/output_data/atlanta_dbscan_map.html"
map_dbscan = create_dbscan_map(df_dbscan, user_weights, cluster_colors=cluster_colors, use_heatmap=True, heatmap_radius=15)
map_dbscan.save(dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...


# Experiment 3 - Sptial DBSCAN

In [175]:
df_dbscan_spatial = dbscan_spatial_clustering(
    df_hexagons,
    eps=0.2,
    min_samples=3,
    spatial_weight=0.4
)


DBSCAN CLUSTERING (Spatially-Aware)
Parameters: eps=0.2, min_samples=3, spatial_weight=0.4

Results:
  Clusters found: 1
  Noise points: 0
  Region 0: 536 hexagons, avg score = 0.377, extent = 33.1 km


In [176]:
cluster_colors_spatial = get_cluster_colors(df_dbscan_spatial)

In [177]:
map_dbscan_spatial = create_dbscan_map(
    df_dbscan_spatial,
    user_weights,
    cluster_colors=cluster_colors_spatial,
    use_heatmap=True,
    heatmap_radius=15
)

spatial_dbscan_map_name = "data/output_data/atlanta_spatial_dbscan_map.html"
map_dbscan_spatial.save(spatial_dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...
